# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [1]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [2]:
import pandas as pd

In [3]:
from google.colab import files
uploaded = files.upload()

Saving reviews_test.csv to reviews_test.csv
Saving reviews_train.csv to reviews_train.csv


In [4]:
train_data = pd.read_csv('reviews_train.csv')
test_data = pd.read_csv('reviews_test.csv')

# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [6]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [7]:
_ = sgd_clf.fit(train_data['review'], train_data['label'])

## Evaluating model accuracy

In [8]:
from sklearn import metrics

In [9]:
def evaluate(clf):
    pred = clf.predict(test_data['review'])
    acc = metrics.accuracy_score(test_data['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [10]:
evaluate(sgd_clf)

Accuracy: 76.8%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

1. Naive Bayes Classifier:

In [11]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline

nb_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB())
])

nb_clf.fit(train_data['review'], train_data['label'])

Pipeline(steps=[('vect', CountVectorizer()), ('tfidf', TfidfTransformer()),
                ('clf', MultinomialNB())])

2. Hyperparameter Tuning (GridSearch):

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'clf__alpha': [0.0001, 0.001, 0.01, 0.1]
}

grid_search = GridSearchCV(nb_clf, param_grid, cv=5, n_jobs=-1)
grid_search.fit(train_data['review'], train_data['label'])

print(f"Best parameters: {grid_search.best_params_}")

Best parameters: {'clf__alpha': 0.1}


3. Ensemble Model (Voting Classifier):

In [13]:
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

svm_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SVC())
])

log_reg_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', LogisticRegression(max_iter=1000))
])

ensemble_clf = VotingClassifier(estimators=[
    ('svm', svm_clf),
    ('logreg', log_reg_clf),
    ('nb', nb_clf)
], voting='hard')

ensemble_clf.fit(train_data['review'], train_data['label'])

VotingClassifier(estimators=[('svm',
                              Pipeline(steps=[('vect', CountVectorizer()),
                                              ('tfidf', TfidfTransformer()),
                                              ('clf', SVC())])),
                             ('logreg',
                              Pipeline(steps=[('vect', CountVectorizer()),
                                              ('tfidf', TfidfTransformer()),
                                              ('clf',
                                               LogisticRegression(max_iter=1000))])),
                             ('nb',
                              Pipeline(steps=[('vect', CountVectorizer()),
                                              ('tfidf', TfidfTransformer()),
                                              ('clf', MultinomialNB())]))])

4. Model Evaluation:

In [14]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate(clf):
    pred = clf.predict(test_data['review'])
    acc = accuracy_score(test_data['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')
    print(confusion_matrix(test_data['label'], pred))
    print(classification_report(test_data['label'], pred))

evaluate(grid_search.best_estimator_)

Accuracy: 80.1%
[[402  98]
 [101 399]]
              precision    recall  f1-score   support

         bad       0.80      0.80      0.80       500
        good       0.80      0.80      0.80       500

    accuracy                           0.80      1000
   macro avg       0.80      0.80      0.80      1000
weighted avg       0.80      0.80      0.80      1000



## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [15]:
train_data.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [16]:
print(train_data.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [17]:
import re

def is_bad_data(review: str) -> bool:
    return bool(re.search(r'<.*?>', review))

bad_data_indices = train_data[train_data['review'].apply(is_bad_data)].index
train_clean = train_data.drop(bad_data_indices)

bad_reviews = train_data.iloc[bad_data_indices]
print(bad_reviews.head())

                                               review label
0   Based on all the negative comments about Taste...  good
2   </tr>The magazine is not worth the cost of sub...  good
5   The magazines are great, but I never received ...  good
10  </div>It's not the fault of the magazine, I ju...  good
11  <li>dispatchEventBest magazine for current and...   bad


## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [18]:
def is_bad_data(review: str) -> bool:
    return bool(re.search(r'<.*?>', review))

## Creating the cleaned training set

In [19]:
train_clean = train_data[~train_data['review'].apply(is_bad_data)]

## Evaluating a model trained on the clean training set

In [20]:
from sklearn import clone

In [21]:
sgd_clf_clean = clone(sgd_clf)

In [22]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [23]:
evaluate(sgd_clf_clean)

Accuracy: 97.0%
[[486  14]
 [ 16 484]]
              precision    recall  f1-score   support

         bad       0.97      0.97      0.97       500
        good       0.97      0.97      0.97       500

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000



# Part 1: Function Design (Core Task)

**Task 1**

In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def compare_models(models, X_train, y_train, X_test, y_test):
    """
    Compare multiple machine learning models on various performance metrics.

    Args:
    models (dict): A dictionary where keys are model names and values are model instances.
    X_train (DataFrame): Training features.
    y_train (Series): Training labels.
    X_test (DataFrame): Test features.
    y_test (Series): Test labels.

    Returns:
    pd.DataFrame: DataFrame containing model names and their performance metrics.
    """
    results = []

    for model_name, model in models.items():
        # Train the model
        model.fit(X_train, y_train)

        # Make predictions
        y_pred = model.predict(X_test)

        # Evaluate performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        # Store the results
        results.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-score': f1
        })

    # Return results as a DataFrame
    return pd.DataFrame(results)